# Method 3 — In-Space Placebo Test

**Logic:** Treat CAPE as the pseudo-treated unit and NATCOR as the donor. Compute its RMSPE ratio. The NATCOR treatment effect is more credible if its ratio exceeds the placebo ratio. With only two units the distribution is thin, but the comparison is the standard SCM inference step.

In [ ]:
import pandas as pd
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

def fit_scm(Y_pre, X_pre):
    if X_pre.ndim == 1:
        X_pre = X_pre.reshape(-1, 1)
    n = X_pre.shape[1]
    w = cp.Variable(n)
    prob = cp.Problem(
        cp.Minimize(cp.sum_squares(Y_pre - X_pre @ w)),
        [w >= 0, cp.sum(w) == 1]
    )
    prob.solve(solver=cp.SCS)
    return np.array(w.value).flatten()

def rmspe(a, b):
    return np.sqrt(np.mean((np.array(a) - np.array(b))**2))


## Data Preparation

In [ ]:
df = pd.read_csv('./data/corridor.csv')
df['Date']        = df['Date'].astype(str).str.strip()
df['Corridor']    = df['Corridor'].astype(str).str.strip()
df['Date_period'] = pd.PeriodIndex(df['Date'], freq='M')

y_col       = 'Rail_Vol_mt'
treated     = 'NATCOR'
treat_start = pd.Period('2022-04', freq='M')
pre_end     = treat_start - 1

wide     = df.pivot(index='Date_period', columns='Corridor', values=y_col).sort_index()
donors   = [c for c in wide.columns if c != treated]
pre_idx  = wide.index[wide.index <= pre_end]
post_idx = wide.index[wide.index >= treat_start]
treat_t  = treat_start.to_timestamp()
t        = wide.index.to_timestamp()

# Base SCM for comparison
w_hat  = fit_scm(wide.loc[pre_idx, treated].values,
                 wide.loc[pre_idx, donors].values)
synth  = pd.Series(wide[donors].values @ w_hat, index=wide.index, name='Synthetic')
actual = wide[treated]
gap    = actual - synth

base_pre_rmspe  = rmspe(actual[pre_idx], synth[pre_idx])
base_post_rmspe = rmspe(actual[post_idx], synth[post_idx])
base_ratio      = base_post_rmspe / base_pre_rmspe

print(f'Pre periods : {len(pre_idx)} ({pre_idx[0]} – {pre_idx[-1]})')
print(f'Post periods: {len(post_idx)} ({post_idx[0]} – {post_idx[-1]})')
print(f'NATCOR Base Ratio: {base_ratio:.4f}')


## In-Space Placebo Estimation

In [ ]:
# ── Method 3 : In-Space Placebo Test ──────────────────────────────────────────
# Logic: Treat CAPE as the pseudo-treated unit; NATCOR becomes the donor.
# Compute CAPE's RMSPE ratio and compare it to NATCOR's.
# If NATCOR's ratio > CAPE's, the effect is more credible.
# Note: with only 2 units, the exact p-value = 1/2 = 0.50 (standard SCM limit).

placebo_unit = 'CAPE'

w_space  = fit_scm(wide.loc[pre_idx, placebo_unit].values,
                   wide.loc[pre_idx, [treated]].values)
sy_space = pd.Series(wide[[treated]].values @ w_space, index=wide.index)
gap_space = wide[placebo_unit] - sy_space

pre_r_sp  = rmspe(wide[placebo_unit][pre_idx], sy_space[pre_idx])
post_r_sp = rmspe(wide[placebo_unit][post_idx], sy_space[post_idx])
ratio_sp  = post_r_sp / pre_r_sp

print(f'NATCOR ratio : {base_ratio:.4f}')
print(f'CAPE placebo : {ratio_sp:.4f}')
distinguishable = base_ratio > ratio_sp
print(f'\nInterpretation: NATCOR ratio {">" if distinguishable else "<="} CAPE ratio')
print(f'  → effect is {"distinguishable" if distinguishable else "NOT distinguishable"} from placebo.')
print(f'  (With only 2 units, exact p-value = 0.50 — standard SCM inference limitation.)')


## Figure 3 — Gap Comparison: NATCOR vs CAPE Placebo

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
fig.suptitle(
    'Figure 3 — In-Space Placebo Test\n'
    '(CAPE as pseudo-treated, NATCOR as donor)',
    fontsize=12, fontweight='bold')

ax = axes[0]
ax.plot(t, wide[placebo_unit].values, color='#1a1a2e', lw=2.2, label='Actual CAPE')
ax.plot(t, sy_space.values,           color='#e94560', lw=2.2, ls='--', label='Synthetic CAPE')
ax.axvline(treat_t, color='grey', lw=1.5, ls=':')
ax.axvspan(treat_t, t[-1], alpha=0.07, color='red')
ax.set_ylabel('Rail Volume (MT)')
ax.legend()
ax.set_title(
    f'CAPE Placebo — Ratio: {ratio_sp:.2f}  vs  NATCOR Ratio: {base_ratio:.2f}',
    fontsize=9, color='#555')
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
ax.plot(t, gap.values,       color='#0f3460', lw=2.5,
        label=f'NATCOR gap  (ratio={base_ratio:.2f})', zorder=3)
ax.plot(t, gap_space.values, color='#e94560', lw=2, ls='--', alpha=0.8,
        label=f'CAPE placebo gap  (ratio={ratio_sp:.2f})')
ax.axhline(0, color='black', lw=0.8)
ax.axvline(treat_t, color='grey', lw=1.5, ls=':')
ax.set_ylabel('Gap (MT)')
ax.set_xlabel('Date')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig3_in_space_placebo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved → fig3_in_space_placebo.png')


## Summary Table

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame([
    {'Unit': 'NATCOR (treated)', 'Pre-RMSPE': round(base_pre_rmspe, 4),
     'Post-RMSPE': round(base_post_rmspe, 4), 'Ratio': round(base_ratio, 4), 'Role': 'Treated'},
    {'Unit': 'CAPE (placebo)',   'Pre-RMSPE': round(pre_r_sp, 4),
     'Post-RMSPE': round(post_r_sp, 4), 'Ratio': round(ratio_sp, 4),  'Role': 'Placebo'},
])
summary
